In [1]:
from PIL import Image
from starvector.model.starvector_arch import StarVectorForCausalLM
from starvector.data.util import process_and_rasterize_svg
import torch

# 8B: SigLIP 384px image encoder + StarCoder2-7B (16k context). Needs ~16-20 GB VRAM in fp16.
# model_name = "starvector/starvector-8b-im2svg"
model_name = "starvector/starvector-1b-im2svg"

starvector = StarVectorForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)

starvector.cuda()
starvector.eval()


You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


trainable params: 0 || all params: 1434095106 || trainable%: 0.0


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

StarVectorForCausalLM(
  (model): StarVectorStarCoder(
    (svg_transformer): StarCoderModel(
      (transformer): GPTBigCodeForCausalLM(
        (transformer): GPTBigCodeModel(
          (wte): Embedding(49156, 2048)
          (wpe): Embedding(8192, 2048)
          (drop): Dropout(p=0.1, inplace=False)
          (h): ModuleList(
            (0-23): 24 x GPTBigCodeBlock(
              (ln_1): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
              (attn): GPTBigCodeFlashAttention2(
                (c_attn): Linear(in_features=2048, out_features=2304, bias=True)
                (c_proj): Linear(in_features=2048, out_features=2048, bias=True)
                (attn_dropout): Dropout(p=0.1, inplace=False)
                (resid_dropout): Dropout(p=0.1, inplace=False)
              )
              (ln_2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
              (mlp): GPTBigCodeMLP(
                (c_fc): Linear(in_features=2048, out_features=8192, bias=True)
     

In [2]:
import re
import numpy as np
from transformers import StoppingCriteria

image_pil = Image.open('../assets/examples/sample-20.png').convert('RGB')

# Use the model's own processor (pads to a white square, like in training).
# starvector.process_images() on the 8B model uses the raw SigLIP processor, which
# stretches non-square images (1002x1200 here) and distorts the drawing.
image = starvector.model.processor(image_pil, return_tensors="pt")["pixel_values"]
if image.dim() == 3:
    image = image.unsqueeze(0)
image = image.cuda()


class LoopStop(StoppingCriteria):
    """Stop generation when the model is stuck repeating the same snippet
    (e.g. 'c0.2,0,0.4,0,0.6,0c0.1,0,0.1,0,0.2,0' forever) instead of wasting the token budget."""
    def __init__(self, tokenizer, every=64, tail_chars=1200, probe_chars=40, max_hits=8):
        self.tok, self.every, self.tail_chars = tokenizer, every, tail_chars
        self.probe_chars, self.max_hits = probe_chars, max_hits
        self.steps, self.triggered = 0, False

    def __call__(self, input_ids, scores, **kwargs):
        self.steps += 1
        if self.steps % self.every:
            return False
        tail = re.sub(r"\s+", "", self.tok.decode(input_ids[0][-700:]))[-self.tail_chars:]
        if len(tail) < self.tail_chars:
            return False
        if tail.count(tail[-self.probe_chars:]) >= self.max_hits:
            self.triggered = True
        return self.triggered


def generate_svg(model, image, max_length=10000, **sampling):
    """Same as model.generate_im2svg, plus loop detection. Returns (raw_svg, looped)."""
    base = model.model
    tok = base.svg_transformer.tokenizer
    inputs_embeds, attention_mask, prompt_tokens = base._prepare_generation_inputs(
        {"image": image}, None, image.device)
    kw = base._get_generation_kwargs({"inputs_embeds": inputs_embeds, "attention_mask": attention_mask,
                                      "max_length": max_length, **sampling})
    kw.update(base._get_im2svg_specific_kwargs({}))
    loop = LoopStop(tok)
    kw["stopping_criteria"].append(loop)
    with torch.inference_mode():
        out = base.svg_transformer.transformer.generate(**kw)
    out = torch.cat([prompt_tokens.input_ids, out], dim=1)
    return tok.batch_decode(out, skip_special_tokens=True)[0], loop.triggered


def repair_truncated_svg(s):
    """Close an SVG that was cut off: drop the unfinished tail element
    and close every still-open tag, so the shapes generated so far are kept."""
    start = s.find("<svg")
    if start < 0:
        return s
    s = s[start:]
    end = s.rfind("</svg>")
    if end >= 0:
        return s[:end + len("</svg>")]
    s = s[:s.rfind(">") + 1]
    stack = []
    for closing, name, selfclose in re.findall(r"<(/?)([A-Za-z][\w:.-]*)[^>]*?(/?)>", s):
        if closing:
            if stack and stack[-1] == name:
                stack.pop()
        elif not selfclose:
            stack.append(name)
    return s + "".join(f"</{n}>" for n in reversed(stack))


def to_square_gray(img, size=256):
    img = img.convert("RGB")
    side = max(img.size)
    canvas = Image.new("RGB", (side, side), "white")
    canvas.paste(img, ((side - img.width) // 2, (side - img.height) // 2))
    return np.asarray(canvas.resize((size, size)).convert("L"), dtype=np.float32) / 255


SHAPE_RE = r"<(path|rect|circle|ellipse|polygon|polyline|line)\b"
target = to_square_gray(image_pil)

# Several sampling attempts. No beam search (num_beams=2 tends to fall into loops).
# Low temperature and (almost) no repetition_penalty: SVG coordinates repeat digits a lot,
# so a strong penalty distorts shapes; LoopStop already handles degenerate loops.
# Each 8B attempt can take several minutes - add more entries if you have the time.
attempts = [
    dict(num_beams=1, temperature=0.6, top_p=0.9,  repetition_penalty=1.0),
    dict(num_beams=1, temperature=0.7, top_p=0.95, repetition_penalty=1.02),
    dict(num_beams=1, temperature=0.5, top_p=0.9,  repetition_penalty=1.03),
]

best = None
for i, params in enumerate(attempts):
    raw_svg, looped = generate_svg(starvector, image, **params)
    with open(f"../assets/sample-20.raw.{i}.svg", "w") as f:
        f.write(raw_svg)
    svg_i, raster_i = process_and_rasterize_svg(repair_truncated_svg(raw_svg))
    n_shapes = len(re.findall(SHAPE_RE, svg_i))
    err = float(np.mean((to_square_gray(raster_i) - target) ** 2)) if n_shapes else float("inf")
    print(f"[{i}] {params} -> {len(raw_svg)} chars, closed={'</svg>' in raw_svg}, "
          f"loop={looped}, shapes={n_shapes}, mse={err:.4f}")
    if n_shapes and (best is None or err < best[0]):
        best = (err, svg_i, raster_i)

if best is None:
    raise RuntimeError("No attempt produced any shape. See ../assets/sample-20.raw.*.svg; "
                       "this image is likely too complex for the 1B model - try the 8B model in cell 1.")
_, svg, raster_image = best


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/opt/conda/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:677: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
../aten/src/ATen/native/cuda/Indexing.cu:1255: indexSelectSmallIndex: block: [7,0,0], thread: [0,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1255: indexSelectSmallIndex: block: [7,0,0], thread: [1,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1255: indexSelectSmallIndex: block: [7,0,0], thread: [2,0,0] Assertion `srcIndex < srcSelectDimSize` failed.
../aten/src/ATen/native/cuda/Indexing.cu:1255: indexSelectSmallIndex: block: [7,0,0], thread: [3,0,0] Ass

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
from IPython.display import display
display(image_pil.resize((256, int(256 * image_pil.height / image_pil.width))), raster_image)
with open("../assets/sample-20.svg", "w") as f:
    f.write(svg)
print("Saved ../assets/sample-20.svg")

In [ ]:
del starvector
torch.cuda.empty_cache()